# Research Assistant

In [3]:
# from dotenv import load_dotenv
# load_dotenv(override=True)

In [4]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_openai langchain_community langchain_core tavily-python wikipedia

## Setup

In [5]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [6]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model='gpt-4o-mini',temperature=0)

## Generate Analysts: Human-In-The-Loop

Create analysts and review them using human-in-the-loop.

In [ ]:
from typing_extensions import TypedDict
from pydantic import BaseModel,Field
from IPython.display import Image, display

class Analyst(BaseModel):
    affiliation:str=Field(
        description='Primary affiliation of the analyst'
    )
    name:str=Field(
        description='Name of the analyst'
    )
    role:str=Field(
        description='Role of the analyst in the context of the topic'
    )
    description:str=Field(
        description="Description of the analyst's focus, concern and motives"
    )
    @property
    def persona(self)->str:
        return (
            f"Name: {self.name}\n"
            f"Role: {self.role}\n"
            f"Affiliation: {self.affiliation}\n"
            f"Description: {self.description}\n"
        )
    
class Perspectives(BaseModel):
    analysts:list[Analyst]=Field(
        description="Comprehensive list of analysts with their roles and affiliations."
    )

class GenerateAnalystState(TypedDict):
    topic:str
    max_analysts:input
    human_analyst_feedback:str
    analysts:list[Analyst]    

In [ ]:
from langgraph.graph import START,END,StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from typing_extensions import Literal

analyst_instructions="""You are tasked with creating a set of AI analyst personas. Follow these instructions carefully:

1. First, review the research topic:
{topic}
        
2. Examine any editorial feedback that has been optionally provided to guide creation of the analysts: 
        
{human_analyst_feedback}
    
3. Determine the most interesting themes based upon documents and / or feedback above.
                    
4. Pick the top {max_analysts} themes.

5. Assign one analyst to each theme."""

def create_analysts(state:GenerateAnalystState):
    topic=state['topic']
    max_analysts=state['max_analysts']
    human_analyst_feedback=state.get('human_analyst_feedback',"")
    structured_llm=llm.with_structured_output(Perspectives)
    
    system_message=analyst_instructions.format(topic=topic,max_analysts=max_analysts,human_analyst_feedback=human_analyst_feedback)
    analysts=structured_llm.invoke([SystemMessage(content=system_message)]+[HumanMessage(content='gnerate the set of analyst')])
    # print(analysts)

    return {'analysts':analysts.analysts}

def human_feedback(state:GenerateAnalystState):
    pass
def should_continue(state:GenerateAnalystState):
    human_analyst_feedback=state.get('human_analyst_feedback',None)
    print(human_analyst_feedback)
    if human_analyst_feedback:
        return 'create_analysts'
    return END


In [ ]:
builder=StateGraph(GenerateAnalystState)
builder.add_node(create_analysts)
builder.add_node(human_feedback)
builder.add_edge(START,'create_analysts')
builder.add_edge('create_analysts','human_feedback')
builder.add_conditional_edges('human_feedback',should_continue,["create_analysts", END])
memory=MemorySaver()
graph=builder.compile(interrupt_before=['human_feedback'],checkpointer=memory)
graph